In [2]:
import torch
import datasets
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

/workspace-SR006.nfs2/bulatov/envs/rmt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RMT

In [3]:
# dataset = datasets.load_from_disk(args.data_path)

dataset_name = "yurakuratov/N8-K2V2-V62_1M"
# dataset_name = "yurakuratov/N8-K1V1-V62_1M"
dataset = datasets.load_dataset(dataset_name)

In [4]:
ds = dataset['train']

In [5]:
ds[0]

{'context': '!V8:Op!!dk:j4!!Qj:5P!!HH:vu!!cR:m7!!yP:7d!!wm:WJ!!I9:dt!|',
 'query': '?!I9:',
 'target': 'dt!|'}

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd/tokenizers/kv_alphabet_62")

In [16]:
# tokenizer.vocab

In [6]:
import sys
sys.path.append("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd")
from modeling_rmt.huggingface import RMTForReasoning, RMTConfig


[2025-09-04 11:25:58,920] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2025-09-04 11:26:01,838] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [7]:
from transformers import AutoConfig, AutoModelForCausalLM
cfg_name = "HuggingFaceTB/SmolLM2-360M"
model_cfg = AutoConfig.from_pretrained(cfg_name)
base_model = AutoModelForCausalLM.from_config(model_cfg)
# base_model

In [25]:
class Holder:
    pass

args = Holder()
args.n_layer = 4
args.n_head = 4
args.n_embd = 128
# args.memory_task = None
args.memory_task = "reconstruct"
# args.memory_task = "retrieval"
args.memory_task_freq = 0.1
args.memory_key_size = 4
args.memory_value_size = 4

In [26]:
from transformers import AutoConfig
from transformers import AutoModelForCausalLM

base_model_config = AutoConfig.from_pretrained('NousResearch/Llama-3.2-1B')
base_model_config.num_hidden_layers = args.n_layer
base_model_config.num_attention_heads = args.n_head
base_model_config.num_key_value_heads = args.n_head
base_model_config.hidden_size = args.n_embd
base_model_config.head_dim = base_model_config.hidden_size // base_model_config.num_attention_heads
base_model_config.intermediate_size = base_model_config.hidden_size * 4

In [27]:
config = RMTConfig()
# config.base_model_name = "HuggingFaceTB/SmolLM2-135M"
config.base_model_config = base_model_config
config.num_mem_tokens = 16
config.max_n_segments = 10
config.think_token_id = 100
config.answer_token_id = 101
config.bos_token_id = 102
config.eos_token_id = 103

model = RMTForReasoning(config)

# model.load_state_dict(torch.load("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd/models/N8-K2V2-V62_1M/model.pt"))

In [40]:
sample['query']

'?!TU:'

In [39]:
context

'!t4:ur!!TU:fI!!WF:Hx!!ZM:PB!!lW:B5!!L8:cE!!WG:4R!!wl:qp!|'

In [69]:
from torch.nn.utils.rnn import pad_sequence
import torch

def collate_fn(batch, memory_task_freq=args.memory_task_freq):
    """
    Collate function that splits each sample into two segments:
    - First segment: context
    - Second segment: query + target
    Pads segments across the batch to the same length.
    """
    def encode(text):
        return tokenizer.encode(text, add_special_tokens=False)

    segments_batch = []
    for sample in batch:
        context = sample['context']
        
        perform_memory_task = torch.rand(1) < memory_task_freq
        if perform_memory_task and args.memory_task == "reconstruct":
            query = '??'
            target = context[:-2]
        elif perform_memory_task and args.memory_task == "continue":
            query_start_ind = torch.randint(0, len(context) - args.memory_key_size - args.memory_value_size - 1, 1)
            query = '!' + context[query_start_ind:query_start_ind + args.memory_key_size]
            target = context[query_start_ind + args.memory_key_size:query_start_ind + args.memory_key_size + args.memory_value_size]
        else:
            query = sample['query']
            target = sample['target']

        # Segment 1: context
        context_ids = encode(context)
        # Segment 2: query + target
        query_ids = encode(query)
        target_ids = encode(target)
        qt_ids = query_ids + target_ids

        # Each segment: dict with input_ids, attention_mask, labels, labels_mask
        # For context segment, no loss (labels = -100)
        seg1 = {
            'input_ids': torch.tensor(context_ids, dtype=torch.long),
            'attention_mask': torch.ones(len(context_ids), dtype=torch.long),
            'labels': torch.full((len(context_ids),), -100, dtype=torch.long),
            'labels_mask': torch.zeros(len(context_ids), dtype=torch.bool)
        }
        # For query+target segment, loss only on target tokens
        qt_input_ids = torch.tensor(qt_ids, dtype=torch.long)
        qt_attention_mask = torch.ones(len(qt_ids), dtype=torch.long)
        # labels: -100 for query, target tokens as labels
        labels = torch.full((len(qt_ids),), -100, dtype=torch.long)
        if len(target_ids) > 0:
            labels[-len(target_ids):] = torch.tensor(target_ids, dtype=torch.long)
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
            labels_mask[-len(target_ids) - 1:] = True
        else:
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
        seg2 = {
            'input_ids': qt_input_ids,
            'attention_mask': qt_attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        segments_batch.append([seg1, seg2])

    # Pad segments across the batch
    batch_segments = []
    num_segments = 2
    id_pad_value = tokenizer.pad_token_id if hasattr(tokenizer, "pad_token_id") and tokenizer.pad_token_id is not None else 0
    for i in range(num_segments):
        input_ids = [s[i]['input_ids'] for s in segments_batch]
        attention_mask = [s[i]['attention_mask'] for s in segments_batch]
        labels = [s[i]['labels'] for s in segments_batch]
        labels_mask = [s[i]['labels_mask'] for s in segments_batch]

        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=id_pad_value)
        attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)
        labels_mask = pad_sequence(labels_mask, batch_first=True, padding_value=False)

        batch_segment = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        batch_segments.append(batch_segment)

    # Concatenate all labels for the batch (for loss computation)
    full_labels = torch.cat([s['labels'] for s in batch_segments], dim=1)
    return {"segments": batch_segments, "labels": full_labels}

In [49]:
batch = [ds[i] for i in range(10)]
collated = collate_fn(batch)


In [50]:
context

'!t4:ur!!TU:fI!!WF:Hx!!ZM:PB!!lW:B5!!L8:cE!!WG:4R!!wl:qp!|'

In [44]:
model.to(dtype=torch.bfloat16)
out = model(**collated)

In [33]:
collated['segments'][0]['input_ids'].shape, collated['segments'][1]['input_ids'].shape

(torch.Size([10, 57]), torch.Size([10, 57]))

In [34]:
out.logits.shape

torch.Size([10, 66, 128256])

In [35]:
out.loss

tensor(5.8125, dtype=torch.bfloat16, grad_fn=<DivBackward0>)

In [78]:
collate_fn_mem = lambda batch: collate_fn(batch, memory_task_freq=1)
collated = collate_fn_mem(batch)


In [79]:
tokenizer.batch_decode(collated['segments'][0]['input_ids'])

['! V 8 : O p ! ! d k : j 4 ! ! Q j : 5 P ! ! H H : v u ! ! c R : m 7 ! ! y P : 7 d ! ! w m : W J ! ! I 9 : d t ! |',
 '! w O : e m ! ! n B : z b ! ! M q : t S ! ! B i : N f ! ! B p : Z p ! ! w M : n t ! ! M P : S j ! ! 5 o : i R ! |',
 '! a O : K x ! ! y A : 6 2 ! ! r O : i S ! ! W i : 1 l ! ! G J : n i ! ! p o : D D ! ! 4 3 : z k ! ! C 6 : 6 i ! |',
 '! S J : W k ! ! L P : 3 D ! ! Q E : y q ! ! E a : G d ! ! N e : e f ! ! u 4 : i x ! ! v F : k x ! ! p Z : h N ! |',
 '! q y : l x ! ! N b : r K ! ! 0 D : O a ! ! 7 f : V r ! ! z Z : x 7 ! ! z N : q 3 ! ! I L : n K ! ! 7 j : 3 Z ! |',
 '! q K : x l ! ! h E : l J ! ! P 9 : Q g ! ! o 6 : D G ! ! K W : 6 w ! ! L z : B W ! ! D j : x l ! ! g n : 4 o ! |',
 '! z V : T k ! ! M T : 7 A ! ! D 2 : k 7 ! ! 7 G : 9 1 ! ! l e : 2 s ! ! c e : 9 g ! ! R e : B u ! ! q z : f r ! |',
 '! a a : B t ! ! S F : q p ! ! U 1 : O F ! ! 6 x : c r ! ! k V : 5 B ! ! Q m : W C ! ! Z u : J 0 ! ! g 5 : 2 R ! |',
 '! f m : Y k ! ! e J : X t ! ! 0 V : W I ! ! p d : q 3 

In [80]:
tokenizer.batch_decode(collated['segments'][1]['input_ids'])

['? ? ! V 8 : O p ! ! d k : j 4 ! ! Q j : 5 P ! ! H H : v u ! ! c R : m 7 ! ! y P : 7 d ! ! w m : W J ! ! I 9 : d t',
 '? ? ! w O : e m ! ! n B : z b ! ! M q : t S ! ! B i : N f ! ! B p : Z p ! ! w M : n t ! ! M P : S j ! ! 5 o : i R',
 '? ? ! a O : K x ! ! y A : 6 2 ! ! r O : i S ! ! W i : 1 l ! ! G J : n i ! ! p o : D D ! ! 4 3 : z k ! ! C 6 : 6 i',
 '? ? ! S J : W k ! ! L P : 3 D ! ! Q E : y q ! ! E a : G d ! ! N e : e f ! ! u 4 : i x ! ! v F : k x ! ! p Z : h N',
 '? ? ! q y : l x ! ! N b : r K ! ! 0 D : O a ! ! 7 f : V r ! ! z Z : x 7 ! ! z N : q 3 ! ! I L : n K ! ! 7 j : 3 Z',
 '? ? ! q K : x l ! ! h E : l J ! ! P 9 : Q g ! ! o 6 : D G ! ! K W : 6 w ! ! L z : B W ! ! D j : x l ! ! g n : 4 o',
 '? ? ! z V : T k ! ! M T : 7 A ! ! D 2 : k 7 ! ! 7 G : 9 1 ! ! l e : 2 s ! ! c e : 9 g ! ! R e : B u ! ! q z : f r',
 '? ? ! a a : B t ! ! S F : q p ! ! U 1 : O F ! ! 6 x : c r ! ! k V : 5 B ! ! Q m : W C ! ! Z u : J 0 ! ! g 5 : 2 R',
 '? ? ! f m : Y k ! ! e J : X t ! ! 0 V : W I ! ! p d : 

In [81]:
collated['segments'][1]['input_ids']

tensor([[67, 67, 66, 51, 64, 68, 44, 19, 66, 66,  7, 14, 68, 13, 60, 66, 66, 46,
         13, 68, 61, 45, 66, 66, 37, 37, 68, 25, 24, 66, 66,  6, 47, 68, 16, 63,
         66, 66, 28, 45, 68, 63,  7, 66, 66, 26, 16, 68, 52, 39, 66, 66, 38, 65,
         68,  7, 23],
        [67, 67, 66, 26, 44, 68,  8, 16, 66, 66, 17, 31, 68, 29,  5, 66, 66, 42,
         20, 68, 23, 48, 66, 66, 31, 12, 68, 43,  9, 66, 66, 31, 19, 68, 55, 19,
         66, 66, 26, 42, 68, 17, 23, 66, 66, 42, 45, 68, 48, 13, 66, 66, 61, 18,
         68, 12, 47],
        [67, 67, 66,  4, 44, 68, 40, 27, 66, 66, 28, 30, 68, 62, 58, 66, 66, 21,
         44, 68, 12, 48, 66, 66, 52, 12, 68, 57, 15, 66, 66, 36, 39, 68, 17, 12,
         66, 66, 19, 18, 68, 33, 33, 66, 66, 60, 59, 68, 29, 14, 66, 66, 32, 62,
         68, 62, 12],
        [67, 67, 66, 48, 39, 68, 52, 14, 66, 66, 41, 45, 68, 59, 33, 66, 66, 46,
         34, 68, 28, 20, 66, 66, 34,  4, 68, 36,  7, 66, 66, 43,  8, 68,  8,  9,
         66, 66, 24, 60, 68, 12, 27, 66, 66

In [48]:
for l, m in zip(collated['segments'][1]['input_ids'], collated['segments'][1]['labels_mask']):
    print(tokenizer.decode(l[m]))


: d t ! |
: n t ! |
: K x ! |
: 3 D ! |
: V r ! |
: Q g ! |
: k 7 ! |
: W C ! |
: W I ! |
: f I ! |
